# K-Means Clustering

## Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score

plt.style.use('seaborn-v0_8-whitegrid')
print('Semua library berhasil diimport.')

## Importing the dataset

In [ ]:
dataset = pd.read_csv('Mall_Customers.csv')
X = dataset.iloc[:, [3, 4]].values

print(f'Shape dataset: {dataset.shape}')
print(f'Jumlah data points: {X.shape[0]}')
print(f'Jumlah fitur: {X.shape[1]}')
print(f'\nKolom: {list(dataset.columns)}')
dataset.head(10)

## Exploratory Data Analysis (EDA)

In [ ]:
print('=== Statistik Deskriptif ===')
print(dataset.describe())

print(f'\n=== Missing Values ===')
print(dataset.isnull().sum())

print(f'\n=== Distribusi Genre ===')
print(dataset['Genre'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(dataset['Annual Income (k$)'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribusi Annual Income', fontsize=13)
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Frekuensi')

axes[1].hist(dataset['Spending Score (1-100)'], bins=15, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Spending Score', fontsize=13)
axes[1].set_xlabel('Spending Score (1-100)')
axes[1].set_ylabel('Frekuensi')

axes[2].scatter(dataset['Annual Income (k$)'], dataset['Spending Score (1-100)'], alpha=0.6, edgecolors='black', linewidth=0.5)
axes[2].set_title('Annual Income vs Spending Score', fontsize=13)
axes[2].set_xlabel('Annual Income (k$)')
axes[2].set_ylabel('Spending Score (1-100)')

plt.tight_layout()
plt.show()

## Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Sebelum scaling:')
print(f'  Mean: {X.mean(axis=0).round(2)}')
print(f'  Std:  {X.std(axis=0).round(2)}')

print('\nSetelah scaling:')
print(f'  Mean: {X_scaled.mean(axis=0).round(4)}')
print(f'  Std:  {X_scaled.std(axis=0).round(4)}')

## Using the elbow method and silhouette score to find the optimal number of clusters

In [ ]:
K_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil)
    print(f'k={k:2d}  |  WCSS: {kmeans.inertia_:10.2f}  |  Silhouette Score: {sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, wcss, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=14)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS (Within-Cluster Sum of Squares)')
axes[0].axvline(x=5, color='r', linestyle='--', alpha=0.7, label='Optimal k=5')
axes[0].legend(fontsize=11)

axes[1].plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score', fontsize=14)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
best_k = list(K_range)[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_k, color='r', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'\nOptimal k berdasarkan Silhouette Score tertinggi: {best_k} (score={max(silhouette_scores):.4f})')

## Training the K-Means model on the dataset

In [ ]:
n_clusters = 5

kmeans = KMeans(
    n_clusters=n_clusters,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)
y_kmeans = kmeans.fit_predict(X_scaled)

dataset['Cluster'] = y_kmeans

print(f'Jumlah cluster: {n_clusters}')
print(f'Centroids (scaled):\n{kmeans.cluster_centers_.round(4)}')
print(f'\nDistribusi data per cluster:')
for i in range(n_clusters):
    count = np.sum(y_kmeans == i)
    print(f'  Cluster {i}: {count} customers ({count/len(y_kmeans)*100:.1f}%)')

## Cluster Analysis & Profiling

In [ ]:
cluster_summary = dataset.groupby('Cluster').agg({
    'Annual Income (k$)': ['mean', 'std', 'min', 'max'],
    'Spending Score (1-100)': ['mean', 'std', 'min', 'max'],
    'Age': ['mean', 'min', 'max'],
    'CustomerID': 'count'
}).round(2)

print('=== Ringkasan Profil per Cluster ===')
print(cluster_summary.to_string())

In [ ]:
cluster_means = dataset.groupby('Cluster')[['Annual Income (k$)', 'Spending Score (1-100)', 'Age']].mean().round(2)

labels_cluster = []
for i in range(n_clusters):
    income = cluster_means.loc[i, 'Annual Income (k$)']
    spending = cluster_means.loc[i, 'Spending Score (1-100)']
    if income < 45 and spending < 45:
        label = 'Low Income - Low Spending'
    elif income < 45 and spending >= 55:
        label = 'Low Income - High Spending'
    elif income >= 55 and spending < 45:
        label = 'High Income - Low Spending'
    elif income >= 55 and spending >= 55:
        label = 'High Income - High Spending'
    else:
        label = 'Moderate Income - Moderate Spending'
    labels_cluster.append(label)
    print(f'Cluster {i}: {label} (Avg Income: {income:.1f}k$, Avg Spending: {spending:.1f}, Avg Age: {cluster_means.loc[i, "Age"]:.1f})')

## Visualising the clusters

In [ ]:
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
markers = ['o', 's', 'D', '^', 'v']

plt.figure(figsize=(12, 8))

for i in range(n_clusters):
    mask = y_kmeans == i
    plt.scatter(
        X[mask, 0], X[mask, 1],
        s=100, c=colors[i], marker=markers[i],
        label=f'Cluster {i} ({labels_cluster[i]})',
        edgecolors='black', linewidth=0.5, alpha=0.8
    )

centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroids_original[:, 0], centroids_original[:, 1], s=250, c='yellow', marker='*', edgecolors='black', linewidth=1.5, label='Centroids', zorder=5)

plt.title('K-Means Clustering - Mall Customers', fontsize=15)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i in range(n_clusters):
    mask = y_kmeans == i
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1], s=80, c=colors[i], label=f'C{i}', edgecolors='black', linewidth=0.5)
axes[0].set_title('Annual Income vs Spending Score\n(Scaled)', fontsize=13)
axes[0].set_xlabel('Annual Income (scaled)')
axes[0].set_ylabel('Spending Score (scaled)')
axes[0].legend()

age_income = dataset[['Age', 'Annual Income (k$)', 'Cluster']].values
for i in range(n_clusters):
    mask = age_income[:, 2] == i
    axes[1].scatter(age_income[mask, 0], age_income[mask, 1], s=80, c=colors[i], label=f'C{i}', edgecolors='black', linewidth=0.5)
axes[1].set_title('Age vs Annual Income', fontsize=13)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Annual Income (k$)')
axes[1].legend()

age_spending = dataset[['Age', 'Spending Score (1-100)', 'Cluster']].values
for i in range(n_clusters):
    mask = age_spending[:, 2] == i
    axes[2].scatter(age_spending[mask, 0], age_spending[mask, 1], s=80, c=colors[i], label=f'C{i}', edgecolors='black', linewidth=0.5)
axes[2].set_title('Age vs Spending Score', fontsize=13)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Spending Score (1-100)')
axes[2].legend()

plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
sil_avg = silhouette_score(X_scaled, y_kmeans)
sil_per_sample = silhouette_samples(X_scaled, y_kmeans)
db_score = davies_bouldin_score(X_scaled, y_kmeans)

print(f'=== Evaluation Metrics (k={n_clusters}) ===')
print(f'Overall Silhouette Score: {sil_avg:.4f}')
print(f'Davies-Bouldin Score: {db_score:.4f}')

print(f'\nSilhouette Score per Cluster:')
for i in range(n_clusters):
    cluster_sil = sil_per_sample[y_kmeans == i]
    print(f'  Cluster {i}: mean={cluster_sil.mean():.4f}, min={cluster_sil.min():.4f}, max={cluster_sil.max():.4f}')

if sil_avg > 0.7:
    print(f'\n>> Kualitas clustering SANGAT BAIK (score > 0.7)')
elif sil_avg > 0.5:
    print(f'\n>> Kualitas clustering BAIK (score > 0.5)')
elif sil_avg > 0.25:
    print(f'\n>> Kualitas clustering CUKUP (score > 0.25)')
else:
    print(f'\n>> Kualitas clustering LEMAH (score <= 0.25)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

y_lower = 10
for i in range(n_clusters):
    cluster_sil_vals = sil_per_sample[y_kmeans == i]
    cluster_sil_vals.sort()

    size_cluster = cluster_sil_vals.shape[0]
    y_upper = y_lower + size_cluster

    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil_vals,
                      facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=2, label=f'Average ({sil_avg:.3f})')
ax.set_title('Silhouette Plot', fontsize=14)
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Perbandingan Init Method pada Data Ini

In [ ]:
init_methods = ['k-means++', 'random']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results = []

for idx, method in enumerate(init_methods):
    ax = axes[idx]
    km = KMeans(n_clusters=n_clusters, init=method, n_init=10, random_state=42)
    labels_temp = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels_temp)
    db = davies_bouldin_score(X_scaled, labels_temp)
    results.append((method, sil, db))

    centroids_temp = scaler.inverse_transform(km.cluster_centers_)
    for i in range(n_clusters):
        mask = labels_temp == i
        ax.scatter(X[mask, 0], X[mask, 1], s=60, c=colors[i], edgecolors='black', linewidth=0.3, alpha=0.7)
    ax.scatter(centroids_temp[:, 0], centroids_temp[:, 1], s=200, c='yellow', marker='*', edgecolors='black', linewidth=1.5)
    ax.set_title(f'{method} (Silhouette: {sil:.4f}, DBI: {db:.4f})', fontsize=13)
    ax.set_xlabel('Annual Income (k$)')
    ax.set_ylabel('Spending Score (1-100)')

plt.suptitle(f'Perbandingan Init Method (k={n_clusters})', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('\n=== Ranking Init Methods ===')
results.sort(key=lambda x: x[1], reverse=True)
for rank, (method, score, dbi) in enumerate(results, 1):
    print(f'  {rank}. {method:12s} -> Silhouette: {score:.4f} | Davies-Bouldin: {dbi:.4f}')

## Analisis dan Kesimpulan

In [ ]:
print('=' * 65)
print('         KESIMPULAN ANALISIS K-MEANS CLUSTERING')
print('=' * 65)
print()
print(f'1. Dataset: Mall Customers ({len(dataset)} customers)')
print(f'2. Fitur yang digunakan: Annual Income & Spending Score')
print(f'3. Feature scaling: StandardScaler (mean=0, std=1)')
print(f'4. Optimal jumlah cluster: {n_clusters}')
print(f'5. Init method terbaik: {results[0][0]} (Silhouette: {results[0][1]:.4f})')
print(f'6. Silhouette Score: {sil_avg:.4f}')
print(f'7. Davies-Bouldin Score: {db_score:.4f}')
print()
print('Profil Cluster yang Ditemukan:')
print('-' * 65)
for i in range(n_clusters):
    n = np.sum(y_kmeans == i)
    inc = dataset[dataset['Cluster'] == i]['Annual Income (k$)'].mean()
    spn = dataset[dataset['Cluster'] == i]['Spending Score (1-100)'].mean()
    age = dataset[dataset['Cluster'] == i]['Age'].mean()
    print(f'  Cluster {i}: {labels_cluster[i]}')
    print(f'    -> {n} customers | Avg Income: {inc:.1f}k$ | Avg Spending: {spn:.1f} | Avg Age: {age:.1f}')

print()
print('Insight Bisnis:')
print('-' * 65)
print('  - Cluster High Income & High Spending: Target utama untuk premium products')
print('  - Cluster High Income & Low Spending: Perlu strategi marketing untuk meningkatkan spending')
print('  - Cluster Low Income & High Spending: Target untuk promosi dan diskon')
print('  - Cluster Low Income & Low Spending: Pasar potensial dengan strategi yang tepat')
print('  - Cluster Moderate: Segment middle-market, cocok untuk loyalty programs')